# Entrenamiento del modelo final
Entrena el modelo final y lo guarda en `modelo_final.joblib`.
Debe contener **solo** el código necesario para replicarlo.

In [1]:
import pandas as pd
import joblib
from pathlib import Path

SEED = 42

## 1. Carga de datos

In [2]:
train = pd.read_json("data/train_data.jsonlines", lines=True)
y = (train["condition"] == "new").astype(int)

## 2. Features

Reproduce las decisiones tomadas en `notebooks/01_eda.ipynb`. Esta parte (aplanado de anidadas, flags) es determinística — no aprende nada de los datos, así que va afuera del `Pipeline`. El capeo de `price`, el encoding y el TF-IDF sí aprenden de los datos y van dentro del `Pipeline` en la sección de Entrenamiento.

In [3]:
from features import aplanar_anidadas

train = aplanar_anidadas(train)
train[["seller_state", "shipping_mode", "n_atributos", "tag_free_relist", "tiene_garantia"]].head()

,seller_state,shipping_mode,n_atributos,tag_free_relist,tiene_garantia
0,Buenos Aires,not_specified,0,False,True
1,Córdoba,not_specified,0,False,True
2,Buenos Aires,not_specified,2,False,False
3,Capital Federal,custom,2,False,True
4,Capital Federal,not_specified,0,False,False


## 3. Entrenamiento

### 3.1 — Sanity check de `CapeoPorMoneda`

Antes de meterla en el `Pipeline`, la pruebo suelta sobre todo `train` para confirmar que reproduce los techos que ya conocíamos del EDA (ARS 180.000 / USD 3.981.440).

In [4]:
from features import CapeoPorMoneda

capeo = CapeoPorMoneda(columnas=["price", "base_price"], columna_grupo="currency_id")
capeo.fit(train)
print(capeo.techos_["price"])

train_capeado = capeo.transform(train)
train_capeado[["price", "base_price"]].describe()

currency_id
ARS     180000.0
USD    3981440.0
Name: price, dtype: float64


,price,base_price
count,7.000000e+04,7.000000e+04
mean,4.030613e+03,4.030496e+03
std,3.702182e+04,3.702183e+04
min,1.000000e+00,1.000000e+00
25%,9.000000e+01,9.000000e+01
50%,2.500000e+02,2.500000e+02
75%,8.000000e+02,8.000000e+02
max,3.981440e+06,3.981440e+06


### 3.2 — Baseline (reproducir la referencia ya medida)

Objetivo: reproducir "regresión logística con `listing_type_id`, `buying_mode`, `log(price)`, log cantidades, `automatic_relist` → accuracy≈0.822, AUC≈0.903" (referencia inicial).

**Supuesto, no confirmado:** la referencia inicial dice "log cantidades" sin especificar cuál. Interpreto que es `log1p(available_quantity)` (una sola, no las dos — son redundantes según el EDA). Si el número no coincide exacto con la referencia, puede ser por esto; lo importante es que quede en el mismo orden de magnitud.

**Qué va a hacer la celda:**
1. `agregar_logs`: función que agrega `log_price` (`log`, porque el mínimo de `price` ya capeado es 1, no hace falta `log1p`) y `log_available_quantity` (`log1p`, porque esta sí tiene ceros).
2. Un `ColumnTransformer` que hace One-Hot de `listing_type_id`/`buying_mode` y deja pasar las numéricas/`automatic_relist` sin tocar.
3. Un `Pipeline` que encadena: capeo → agregar logs → encoding → `LogisticRegression`. El capeo va primero porque `log_price` necesita el precio ya capeado.

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from features import agregar_logs

columnas_baseline = ["log_price", "log_available_quantity", "automatic_relist"]

preprocesador_baseline = ColumnTransformer([
    ("onehot", OneHotEncoder(handle_unknown="ignore"), ["listing_type_id", "buying_mode"]),
    ("num", "passthrough", columnas_baseline),
])

pipeline_baseline = Pipeline([
    ("capeo", CapeoPorMoneda(columnas=["price", "base_price"], columna_grupo="currency_id")),
    ("logs", FunctionTransformer(agregar_logs)),
    ("preprocesador", preprocesador_baseline),
    ("modelo", LogisticRegression(max_iter=1000, random_state=SEED)),
])

pipeline_baseline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('capeo', ...), ('logs', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,columnas,"['price', 'base_price']"
,columna_grupo,'currency_id'
,percentil,0.995
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function agr...t 0x35b089120>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False


In [6]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
resultados_baseline = cross_validate(pipeline_baseline, train, y, cv=cv, scoring=["accuracy", "roc_auc"])

print("accuracy:", resultados_baseline["test_accuracy"].mean().round(3), "+/-", resultados_baseline["test_accuracy"].std().round(3))
print("roc_auc: ", resultados_baseline["test_roc_auc"].mean().round(3), "+/-", resultados_baseline["test_roc_auc"].std().round(3))
print("\nReferencia (CLAUDE.md): accuracy≈0.822, AUC≈0.903")

accuracy: 0.813 +/- 0.002
roc_auc:  0.896 +/- 0.002

Referencia (CLAUDE.md): accuracy≈0.822, AUC≈0.903


### 3.3 — Un inciso por modelo, con ajuste de hiperparámetros

Para cada modelo del material (regresión logística, LDA, QDA, Naive Bayes Gaussiano, kNN, SVM, árbol de decisión, Random Forest, Extra Trees, AdaBoost, Gradient Boosting) hago un inciso separado: qué hiperparámetro(s) tunea, por qué, y un `GridSearchCV` sobre el mismo `Pipeline` (mismo capeo, mismos logs, mismo encoding/escalado) y la misma CV de 5 folds. Elijo el mejor por `roc_auc` (`refit="roc_auc"`), pero guardo también el `accuracy` de esa misma combinación para la tabla comparativa del final.

Guardo cada mejor modelo ya ajustado en el diccionario `modelos_ajustados`, para no tener que reentrenar nada en la comparación final ni en `predict.ipynb` más adelante.

**Features (actualizado tras `notebooks/02_features.ipynb`, `03_titulo.ipynb` y `04_vendedor.ipynb`):** además de las 5 originales, sumo los bloques B (cantidades), C (contenido de la publicación), D (envío y pago) y E (vendedor y precio) — todos con ganancia de accuracy claramente por encima del desvío entre folds. Dejo afuera el bloque F (tags/estado), que midió ganancia dentro del ruido. Sumo también **TF-IDF del título** (`max_features=500`, el aporte individual más grande junto con el vendedor) y **`VendedorEncoder`** (agregados de `seller_id`: cantidad de publicaciones, precio promedio, y `seller_pct_new` con *leave-one-out* para evitar leakage). El preprocesador sigue siendo One-Hot con `drop="first"` para las categóricas y `StandardScaler` para las numéricas (las columnas booleanas pasan sin tocar). Con ~530 columnas, fuerzo salida densa (`sparse_threshold=0`) porque `HistGradientBoostingClassifier` no acepta matrices sparse.

**Actualizado tras `notebooks/06_eda_nuevas.ipynb` y `07_nuevas_features.ipynb`:** sumo `category_id` codificada con `TargetEncoder(target_type="binary")` (cross-fitting + *smoothing*, no el *leave-one-out* de `VendedorEncoder` — con 75% de categorías con 5 publicaciones o menos, el *leave-one-out* sufre el problema de inversión en grupos chicos). Midió +0,0049 de accuracy con CV, claramente por encima del desvío entre folds (~0,002) — el único de los tres bloques nuevos (categoría, fechas, precio) con ganancia sólida.

**Nota — quedan afuera por ahora:** `07_nuevas_features.ipynb` también midió `hora_inicio`/`gap_actualizacion` (fechas, +0,0018) y flags de redondez de `price` (+0,0016). Ambas ganancias están al límite del desvío entre folds (aporte real pero marginal, probablemente solapado con título/vendedor). Quedan anotadas como posible mejora futura si hace falta exprimir más accuracy, pero no se agregan ahora.

In [7]:
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from features import VendedorEncoder
import numpy as np
import warnings

preprocesador_comparacion = ColumnTransformer([
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first"), [
        "listing_type_id", "buying_mode", "shipping_mode", "seller_state",
    ]),
    ("escala", StandardScaler(), [
        "log_price", "log_available_quantity",
        "sold_quantity", "initial_quantity",
        "n_fotos", "n_atributos", "n_variaciones", "n_metodos_pago",
    ]),
    ("bool", "passthrough", [
        "automatic_relist",
        "tiene_video", "tiene_garantia",
        "free_shipping", "local_pick_up", "accepts_mercadopago",
        "es_tienda_oficial", "tiene_descuento", "tiene_oferta", "tiene_parent_item",
    ]),
    ("tfidf", TfidfVectorizer(max_features=500), "title"),
    ("vendedor", VendedorEncoder(incluir_target=True), ["seller_id", "log_price"]),
    ("categoria", TargetEncoder(target_type="binary", random_state=SEED), ["category_id"]),
], sparse_threshold=0)

modelos_ajustados = {}
filas_comparacion = []


def ajustar_y_registrar(nombre, modelo, param_grid, n_jobs=-1):
    """Arma el Pipeline completo, corre GridSearchCV (refit por roc_auc) y
    guarda el mejor estimador + sus métricas de CV. `n_jobs` se puede bajar
    para modelos que usan mucha memoria por proceso paralelo (ej. kNN)."""
    pipeline = Pipeline([
        ("capeo", CapeoPorMoneda(columnas=["price", "base_price"], columna_grupo="currency_id")),
        ("logs", FunctionTransformer(agregar_logs)),
        ("preprocesador", preprocesador_comparacion),
        ("modelo", modelo),
    ])
    grid = GridSearchCV(pipeline, param_grid, scoring=["accuracy", "roc_auc"], refit="roc_auc", cv=cv, n_jobs=n_jobs, verbose=1)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        grid.fit(train, y)

    modelos_ajustados[nombre] = grid.best_estimator_
    filas_comparacion.append({
        "modelo": nombre,
        "mejores_params": grid.best_params_,
        "accuracy": grid.cv_results_["mean_test_accuracy"][grid.best_index_],
        "roc_auc": grid.cv_results_["mean_test_roc_auc"][grid.best_index_],
    })
    print(nombre, "-> mejores params:", grid.best_params_)
    print("   accuracy:", round(filas_comparacion[-1]["accuracy"], 3), "| roc_auc:", round(filas_comparacion[-1]["roc_auc"], 3))
    return grid

#### 3.3.1 — Regresión logística (L1/L2)

Hiperparámetros: `penalty` (L1 o L2, `Continuación de regresion y Regulización L1 y L2.pdf`) y `C` (inverso de la fuerza de regularización — `C` chico = regulariza más). Uso `solver="liblinear"` porque es el único solver de sklearn que soporta L1 y L2 en el mismo objeto.

**Nota de memoria:** uso `n_jobs=4` (no `-1`) — con ~530 columnas densas, varios procesos en paralelo copiando esa matriz pueden agotar la RAM (mismo motivo que kNN). La corrida anterior de esta celda quedó interrumpida a mano (`KeyboardInterrupt`), por eso no había resultado.

In [8]:
param_grid_logreg = {
    "modelo__penalty": ["l1", "l2"],
    "modelo__C": [0.01, 0.1, 1, 10, 100],
}

grid_logreg = ajustar_y_registrar(
    "Regresión logística",
    LogisticRegression(max_iter=1000, random_state=SEED, solver="liblinear"),
    param_grid_logreg,
    n_jobs=4,
)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Regresión logística -> mejores params: {'modelo__C': 1, 'modelo__penalty': 'l1'}
   accuracy: 0.913 | roc_auc: 0.971


#### 3.3.2 — LDA

Hiperparámetro: `shrinkage` (`Presentación_NB_LDA_QDA.pdf`) — regulariza la matriz de covarianza compartida hacia una diagonal, útil cuando hay colinealidad entre features (como nuestros dummies). Requiere `solver="lsqr"` o `"eigen"` (el solver por defecto, `"svd"`, no soporta `shrinkage`).

In [9]:
param_grid_lda = {
    "modelo__solver": ["lsqr", "eigen"],
    "modelo__shrinkage": [None, "auto", 0.1, 0.5, 0.9],
}

grid_lda = ajustar_y_registrar("LDA", LinearDiscriminantAnalysis(), param_grid_lda)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
LDA -> mejores params: {'modelo__shrinkage': 'auto', 'modelo__solver': 'lsqr'}
   accuracy: 0.912 | roc_auc: 0.97


#### 3.3.3 — QDA

Hiperparámetro: `reg_param` — ya vimos en la comparación rápida que sin regularizar, QDA rompe (covarianza singular dentro de una clase por la colinealidad de los dummies). Barro un rango de valores en vez de usar el `0.1` fijo que probé antes.

In [10]:
param_grid_qda = {
    "modelo__reg_param": [0.0, 0.05, 0.1, 0.2, 0.3, 0.5],
}

grid_qda = ajustar_y_registrar("QDA", QuadraticDiscriminantAnalysis(), param_grid_qda)

Fitting 5 folds for each of 6 candidates, totalling 30 fits
QDA -> mejores params: {'modelo__reg_param': 0.1}
   accuracy: 0.721 | roc_auc: 0.949


#### 3.3.4 — Naive Bayes Gaussiano

Hiperparámetro: `var_smoothing` — un valor chiquito que se suma a la varianza de cada feature para estabilizar el cálculo cuando alguna varianza da casi 0 (nuestros dummies 0/1 son un caso típico). Pruebo un rango logarítmico, como recomienda la documentación de sklearn.

In [11]:
param_grid_nb = {
    "modelo__var_smoothing": np.logspace(-9, -1, 9),
}

grid_nb = ajustar_y_registrar("Naive Bayes (Gaussiano)", GaussianNB(), param_grid_nb)

Fitting 5 folds for each of 9 candidates, totalling 45 fits
Naive Bayes (Gaussiano) -> mejores params: {'modelo__var_smoothing': np.float64(0.1)}
   accuracy: 0.643 | roc_auc: 0.946


#### 3.3.5 — kNN

Hiperparámetros: `n_neighbors` (*k* — el material muestra que pocos vecinos sobreajusta y muchos subajusta) y `weights` (uniforme vs. pesado por distancia, `Presentacion_kNN.pdf`). Ya viene con `StandardScaler` en el preprocesador compartido.

**Nota de memoria:** uso `n_jobs=1` (antes `2`, ahora bajado de nuevo) — con ~530 columnas (500 son del TF-IDF) calcular distancias en paralelo agotaba la RAM y tiraba `BrokenProcessPool`.

In [12]:
param_grid_knn = {
    "modelo__n_neighbors": [5, 9, 15, 21, 31],
    "modelo__weights": ["uniform", "distance"],
}

grid_knn = ajustar_y_registrar("kNN", KNeighborsClassifier(), param_grid_knn, n_jobs=1)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
kNN -> mejores params: {'modelo__n_neighbors': 31, 'modelo__weights': 'distance'}
   accuracy: 0.898 | roc_auc: 0.96


#### 3.3.6 — SVM lineal

Hiperparámetro: `C` (`SVM (1).pdf`) — controla el margen suave: `C` chico permite más errores de clasificación (margen más ancho), `C` grande castiga más los errores (margen más angosto, más parecido al margen máximo original). Sigo con `LinearSVC` por el mismo motivo que antes (kernel no lineal sería muy costoso con 70.000 filas).

In [13]:
param_grid_svm = {
    "modelo__C": [0.01, 0.1, 1, 10],
}

grid_svm = ajustar_y_registrar("SVM lineal", LinearSVC(max_iter=5000, random_state=SEED), param_grid_svm)

Fitting 5 folds for each of 4 candidates, totalling 20 fits
SVM lineal -> mejores params: {'modelo__C': 0.1}
   accuracy: 0.913 | roc_auc: 0.971


#### 3.3.7 — Árbol de decisión (CART)

Nuevo en el material (`Árboles.pdf`), no estaba en la tabla original.

Hiperparámetros: `criterion` (impureza para elegir cada partición — Gini o Entropía, ambas en el material) y `max_depth` (tamaño máximo del árbol, "Tamaño del árbol"). Sumo también `ccp_alpha` (costo de complejidad: `Cα(T) = Σ N_m·Q_m(T) + α|T|` — penaliza árboles con muchos nodos, es la poda tipo *weakest link pruning* que muestra el material) para no depender solo de `max_depth` como forma de controlar el sobreajuste.

In [14]:
param_grid_arbol = {
    "modelo__criterion": ["gini", "entropy"],
    "modelo__max_depth": [3, 5, 10, None],
    "modelo__ccp_alpha": [0.0, 0.001, 0.01],
}

grid_arbol = ajustar_y_registrar("Árbol de decisión", DecisionTreeClassifier(random_state=SEED), param_grid_arbol)

Fitting 5 folds for each of 24 candidates, totalling 120 fits
Árbol de decisión -> mejores params: {'modelo__ccp_alpha': 0.001, 'modelo__criterion': 'entropy', 'modelo__max_depth': 10}
   accuracy: 0.908 | roc_auc: 0.967


#### 3.3.8 — Random Forest

Hiperparámetros: `n_estimators` (cantidad de árboles del bosque) y `max_depth` (tamaño máximo de cada árbol). Dejo `max_features="sqrt"` fijo (no lo tuneo) — es el valor que el material recomienda como típico para Random Forest (`m ≈ √p`); probar también `max_features=None` (todas las features, equivalente a Bagging) resultó carísimo en tiempo de cómputo con 70.000 filas, sin ganancia clara.

In [15]:
param_grid_rf = {
    "modelo__n_estimators": [100, 200],
    "modelo__max_depth": [10, 20],
}

grid_rf = ajustar_y_registrar(
    "Random Forest",
    RandomForestClassifier(max_features="sqrt", random_state=SEED),
    param_grid_rf,
)

Fitting 5 folds for each of 4 candidates, totalling 20 fits
Random Forest -> mejores params: {'modelo__max_depth': 20, 'modelo__n_estimators': 200}
   accuracy: 0.918 | roc_auc: 0.972


#### 3.3.9 — Extra Trees

Misma idea que Random Forest, mismos hiperparámetros (`n_estimators`, `max_depth`) y mismo `max_features="sqrt"` fijo por el mismo motivo de tiempo de cómputo. La diferencia de Extra Trees es que el punto de corte de cada partición también se elige al azar entre varias opciones, en vez de buscar el óptimo — el material lo describe como "similar al Random Forest (sample de observaciones y subconjunto de características)".

In [16]:
param_grid_extra_trees = {
    "modelo__n_estimators": [100, 200],
    "modelo__max_depth": [10, 20],
}

grid_extra_trees = ajustar_y_registrar(
    "Extra Trees",
    ExtraTreesClassifier(max_features="sqrt", random_state=SEED),
    param_grid_extra_trees,
)

Fitting 5 folds for each of 4 candidates, totalling 20 fits
Extra Trees -> mejores params: {'modelo__max_depth': 20, 'modelo__n_estimators': 200}
   accuracy: 0.913 | roc_auc: 0.971


#### 3.3.10 — AdaBoost

Hiperparámetros: `n_estimators` (cantidad de árboles chicos que se van agregando secuencialmente, cada uno corrigiendo al anterior) y `learning_rate` (cuánto pesa cada árbol nuevo en la suma final). Es la técnica *AdaBoost.M1* del material: en cada paso le da más peso a las observaciones mal clasificadas por el árbol anterior.

In [17]:
param_grid_adaboost = {
    "modelo__n_estimators": [50, 100, 200],
    "modelo__learning_rate": [0.1, 0.5, 1.0],
}

grid_adaboost = ajustar_y_registrar("AdaBoost", AdaBoostClassifier(random_state=SEED), param_grid_adaboost)

Fitting 5 folds for each of 9 candidates, totalling 45 fits
AdaBoost -> mejores params: {'modelo__learning_rate': 1.0, 'modelo__n_estimators': 200}
   accuracy: 0.911 | roc_auc: 0.969


#### 3.3.11 — Gradient Boosting

**Aclaración:** uso `HistGradientBoostingClassifier` en vez del `GradientBoostingClassifier` "clásico" — el material dice explícitamente que el Histogram Gradient Boosting "es un Gradient Boosting para grandes cantidades de datos" (agrupa los valores en histogramas en vez de mirar cada dato puntual), y con 70.000 filas entra en ese caso.

Hiperparámetros: `max_iter` (cantidad de árboles/iteraciones del boosting), `learning_rate` (cuánto corrige cada iteración) y `max_depth` (tamaño de cada árbol individual, que en boosting suele ser chico a propósito).

In [18]:
param_grid_histgb = {
    "modelo__max_iter": [100, 200],
    "modelo__learning_rate": [0.05, 0.1],
    "modelo__max_depth": [None, 5],
}

grid_histgb = ajustar_y_registrar("Gradient Boosting", HistGradientBoostingClassifier(random_state=SEED), param_grid_histgb)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Gradient Boosting -> mejores params: {'modelo__learning_rate': 0.05, 'modelo__max_depth': None, 'modelo__max_iter': 200}
   accuracy: 0.924 | roc_auc: 0.978


#### 3.3.12 — Ensamble: Voting (soft)

Del material de la materia SIA (`Voting / Stacking`, aplica igual acá): combino las probabilidades de los modelos ya tuneados en `modelos_ajustados`, **excluyendo**:
- `SVM lineal` (`LinearSVC` no tiene `predict_proba`, y el voto "soft" promedia probabilidades, no clases).
- `QDA` y `Naive Bayes (Gaussiano)` (72-73% de accuracy, muy por debajo del resto que ronda 80-83%) — un promedio simple con modelos tan flojos adentro puede arrastrar para abajo al ensamble.

Reutilizo cada modelo con sus mejores hiperparámetros ya encontrados (no vuelvo a tunear nada acá, solo los combino) y mido el ensamble con la misma CV de 5 folds y el mismo scoring que el resto, para que sea comparable en la tabla final.

In [19]:
from sklearn.ensemble import VotingClassifier

excluidos_voting = {"SVM lineal", "QDA", "Naive Bayes (Gaussiano)"}

estimadores_voting = [
    (nombre, modelos_ajustados[nombre].named_steps["modelo"])
    for nombre in modelos_ajustados
    if nombre not in excluidos_voting
]
print("Modelos en el Voting:", [nombre for nombre, _ in estimadores_voting])

pipeline_voting_soft = Pipeline([
    ("capeo", CapeoPorMoneda(columnas=["price", "base_price"], columna_grupo="currency_id")),
    ("logs", FunctionTransformer(agregar_logs)),
    ("preprocesador", preprocesador_comparacion),
    ("modelo", VotingClassifier(estimators=estimadores_voting, voting="soft")),
])

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    resultados_voting_soft = cross_validate(pipeline_voting_soft, train, y, cv=cv, scoring=["accuracy", "roc_auc"])
    pipeline_voting_soft.fit(train, y)

modelos_ajustados["Voting (soft)"] = pipeline_voting_soft
filas_comparacion.append({
    "modelo": "Voting (soft)",
    "mejores_params": {"estimators": [nombre for nombre, _ in estimadores_voting]},
    "accuracy": resultados_voting_soft["test_accuracy"].mean(),
    "roc_auc": resultados_voting_soft["test_roc_auc"].mean(),
})
print("Voting (soft) -> accuracy:", round(filas_comparacion[-1]["accuracy"], 3), "| roc_auc:", round(filas_comparacion[-1]["roc_auc"], 3))

Modelos en el Voting: ['Regresión logística', 'LDA', 'kNN', 'Árbol de decisión', 'Random Forest', 'Extra Trees', 'AdaBoost', 'Gradient Boosting']
Voting (soft) -> accuracy: 0.919 | roc_auc: 0.976


#### 3.3.13 — Ensamble: Voting (hard)

Voto por clase (mayoría), no por probabilidad — por eso ahora sí puedo sumar `SVM lineal` (no necesita `predict_proba`). Sigo excluyendo `QDA` y `Naive Bayes (Gaussiano)` por el mismo motivo que antes (accuracy muy baja), y también excluyo `Voting (soft)` para no meter un ensamble adentro de otro.

**Límite de esta métrica:** `VotingClassifier(voting="hard")` no expone `predict_proba` ni `decision_function`, así que no se le puede calcular `roc_auc` — solo mido `accuracy`.

In [20]:
excluidos_voting_hard = {"QDA", "Naive Bayes (Gaussiano)", "Voting (soft)"}

estimadores_voting_hard = [
    (nombre, modelos_ajustados[nombre].named_steps["modelo"])
    for nombre in modelos_ajustados
    if nombre not in excluidos_voting_hard
]
print("Modelos en el Voting (hard):", [nombre for nombre, _ in estimadores_voting_hard])

pipeline_voting_hard = Pipeline([
    ("capeo", CapeoPorMoneda(columnas=["price", "base_price"], columna_grupo="currency_id")),
    ("logs", FunctionTransformer(agregar_logs)),
    ("preprocesador", preprocesador_comparacion),
    ("modelo", VotingClassifier(estimators=estimadores_voting_hard, voting="hard")),
])

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    resultados_voting_hard = cross_validate(pipeline_voting_hard, train, y, cv=cv, scoring=["accuracy"])
    pipeline_voting_hard.fit(train, y)

modelos_ajustados["Voting (hard)"] = pipeline_voting_hard
filas_comparacion.append({
    "modelo": "Voting (hard)",
    "mejores_params": {"estimators": [nombre for nombre, _ in estimadores_voting_hard]},
    "accuracy": resultados_voting_hard["test_accuracy"].mean(),
    "roc_auc": np.nan,
})
print("Voting (hard) -> accuracy:", round(filas_comparacion[-1]["accuracy"], 3), "| roc_auc: N/A (no tiene predict_proba)")

Modelos en el Voting (hard): ['Regresión logística', 'LDA', 'kNN', 'SVM lineal', 'Árbol de decisión', 'Random Forest', 'Extra Trees', 'AdaBoost', 'Gradient Boosting']
Voting (hard) -> accuracy: 0.918 | roc_auc: N/A (no tiene predict_proba)


#### 3.3.14 — Ensamble: Stacking

A diferencia del Voting (que promedia u vota con pesos fijos), acá un **meta-modelo** (`LogisticRegression`) aprende a combinar las predicciones de los modelos base. Para que ese aprendizaje no haga *data leakage*, `StackingClassifier` genera esas predicciones con su **propia CV interna** — por eso, para no terminar con una CV anidada carísima (5 de afuera × 5 de adentro × cantidad de modelos), la acoto a `cv=3` interno y uso solo los **4 modelos con mejor `roc_auc`** de la tabla (`Gradient Boosting`, `Random Forest`, `kNN`, `Árbol de decisión`) en vez de los 8 del Voting.

In [21]:
from sklearn.ensemble import StackingClassifier

modelos_para_stacking = ["Gradient Boosting", "Random Forest", "kNN", "Árbol de decisión"]

estimadores_stacking = [
    (nombre, modelos_ajustados[nombre].named_steps["modelo"])
    for nombre in modelos_para_stacking
]

pipeline_stacking = Pipeline([
    ("capeo", CapeoPorMoneda(columnas=["price", "base_price"], columna_grupo="currency_id")),
    ("logs", FunctionTransformer(agregar_logs)),
    ("preprocesador", preprocesador_comparacion),
    ("modelo", StackingClassifier(
        estimators=estimadores_stacking,
        final_estimator=LogisticRegression(max_iter=1000, random_state=SEED),
        cv=3,
    )),
])

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    resultados_stacking = cross_validate(pipeline_stacking, train, y, cv=cv, scoring=["accuracy", "roc_auc"])
    pipeline_stacking.fit(train, y)

modelos_ajustados["Stacking"] = pipeline_stacking
filas_comparacion.append({
    "modelo": "Stacking",
    "mejores_params": {"estimators": modelos_para_stacking, "final_estimator": "LogisticRegression", "cv_interno": 3},
    "accuracy": resultados_stacking["test_accuracy"].mean(),
    "roc_auc": resultados_stacking["test_roc_auc"].mean(),
})
print("Stacking -> accuracy:", round(filas_comparacion[-1]["accuracy"], 3), "| roc_auc:", round(filas_comparacion[-1]["roc_auc"], 3))

Stacking -> accuracy: 0.924 | roc_auc: 0.977


### 3.4 — Comparación final (métricas de CV en train)

Tabla con el mejor resultado de cada modelo ya tuneado, más los 3 ensambles (Voting soft, Voting hard, Stacking). `modelos_ajustados` queda con los 14 `Pipeline` completos listos para reusar en `predict.ipynb` sin volver a ajustar nada.

In [22]:
comparacion_final = pd.DataFrame(filas_comparacion).set_index("modelo").sort_values("roc_auc", ascending=False)
comparacion_final = comparacion_final.sort_values("accuracy", ascending=False)
print(comparacion_final)

                                                            mejores_params  \
modelo                                                                       
Stacking                 {'estimators': ['Gradient Boosting', 'Random F...   
Gradient Boosting        {'modelo__learning_rate': 0.05, 'modelo__max_d...   
Voting (soft)            {'estimators': ['Regresión logística', 'LDA', ...   
Voting (hard)            {'estimators': ['Regresión logística', 'LDA', ...   
Random Forest            {'modelo__max_depth': 20, 'modelo__n_estimator...   
Regresión logística              {'modelo__C': 1, 'modelo__penalty': 'l1'}   
SVM lineal                                              {'modelo__C': 0.1}   
Extra Trees              {'modelo__max_depth': 20, 'modelo__n_estimator...   
LDA                      {'modelo__shrinkage': 'auto', 'modelo__solver'...   
AdaBoost                 {'modelo__learning_rate': 1.0, 'modelo__n_esti...   
Árbol de decisión        {'modelo__ccp_alpha': 0.001, 'modelo__c

## 4. Guardado

In [23]:
joblib.dump(modelos_ajustados, "modelos_ajustados.joblib")

['modelos_ajustados.joblib']